# [실습1] 엑셀 데이터를 활용하는 챗봇의 개선 가능성

## 실습 목표
---
엑셀 데이터를 단순히 활용하는 것으로 대처할 수 없는 질문과 그에 대한 챗봇의 답변을 살펴보고, 이를 머신러닝 기법을 활용해 어느 정도 대처할 수 있음을 이해합니다.

## 실습 목차
---

1. **데이터 활용 챗봇 구성:** 1일차 4챕터에서 구현한 엑셀 데이터를 활용하는 챗봇을 다시 구성합니다.

2. **현재 챗봇의 한계점:** 단순 데이터 활용 챗봇이 대응하기 어려운 질문과 이를 해결할 수 있는 방법을 논의합니다.

3. **주어진 질문을 위한 머신러닝을 활용한 데이터 분석:** 머신러닝을 통해 질문에 답할 수 있는 분석 결과를 반환하는 모델을 학습할 수 있음을 이해합니다.

## 실습 개요
---
엑셀 데이터를 단순히 활용하는 것으로 대처할 수 없는 질문을 머신러닝 모델을 활용해 답변합니다.

## 0. 환경 설정
- 필요한 라이브러리를 불러옵니다.

In [ ]:
import contextlib
import io
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict

- Ollama를 통해 Mistral 7B 모델을 불러옵니다.

In [ ]:
!ollama pull mistral:7b

## 1. 데이터 활용 챗봇 구성
   1일차 4챕터에서 구현한 엑셀 데이터를 활용하는 챗봇을 다시 구성합니다.

먼저, mistral:7b 모델을 사용하는 ChatOllama 객체를 생성하고, 데이터를 불러옵니다.

In [ ]:
llm = ChatOllama(model="mistral:7b")

# 데이터를 불러오고, 이름과 컬럼명을 저장합니다.
data_dir = './data'
df_inkjet = pd.read_csv(os.path.join(data_dir, 'InkjetDB_preprocessing.csv'), index_col=0)

# 데이터를 저장한 변수명을 LLM에 제공하여 이 변수를 활용하는 코드를 작성하게 할 수 있습니다.
df_name = "df_inkjet"
df_columns = ", ".join(df_inkjet.columns)

코드를 파싱하고 실행하는 함수를 정의합니다.

In [ ]:
# LLM이 생성한 코드를 파싱하는 함수를 정의합니다.
def python_code_parser(input: str) -> str:
    # LLM은 대부분 ``` 블럭 안에 코드를 출력합니다. 이를 활용합니다.
    # ```python (코드) ```, 혹은 ``` (코드) ``` 형태로 출력됩니다. 두 경우 모두에 대응하도록 코드를 작성합니다.
    processed_input = input.replace("```python", "```").strip()
    parsed_input_list = processed_input.split("```")

    # 만약 ``` 블럭이 없다면, 입력 텍스트 전체가 코드라고 간주합니다.
    # 아닐 경우 이어지는 코드 실행 과정에서 예외 처리를 통해 오류를 확인할 수 있습니다.
    if len(parsed_input_list) == 1:
        return processed_input

    # 코드 부분만 추출합니다. 
    # LLM은 여러 코드 블럭에 걸쳐 필요한 코드를 출력할 수 있으므로, 코드가 있는 홀수 번째 텍스트를 모두 저장합니다.
    parsed_code_list = []
    for i in range(1, len(parsed_input_list), 2):
        parsed_code_list.append(parsed_input_list[i])
    
    # 코드 부분을 하나로 합칩니다.
    return "\n".join(parsed_code_list)

# 생성한 코드를 실행하는 함수를 정의합니다.
def run_code(input_code: str):
    # 코드가 출력한 값을 캡쳐하기 위한 StringIO 객체를 생성합니다.
    output = io.StringIO()
    try:
        # Redirect stdout to the StringIO object
        with contextlib.redirect_stdout(output):
            # Python 3.10 버전이므로, 키워드 인자를 사용할 수 없습니다.
            # 코드가 실행하면서 출력한 모든 결과를 캡쳐합니다.
            exec(input_code, {"df_inkjet": df_inkjet})
    except Exception as e:
        # 에러가 발생할 경우, 이를 StringIO 객체에 저장합니다.
        print(f"Error: {e}", file=output)
    # StringIO 객체에 저장된 값을 반환합니다.
    return output.getvalue()

`StateGraph` 객체를 생성합니다.

In [ ]:
class State(TypedDict):
    # 그래프 상태의 속성을 정의합니다.
    # 질문, LLM이 생성한 텍스트, 데이터, 코드를 저장합니다.
    question: str
    generation: str
    data: str
    code: str
        
# 그래프를 구성하기 위해 StateGraph 객체를 생성합니다.
# 생성자의 인자로 State를 전달하여 Node 간에 정보를 전달할 때 State type을 사용함을 명시합니다.
workflow = StateGraph(State)

Node에 대응하는 함수를 정의합니다.

In [ ]:
def find_data(question: str) -> str:
    """입력한 질문을 위한 데이터 쿼리 코드를 생성하여 실행한 후 결과를 반환하는 Tool."""
        # 프롬프트를 정의합니다.
    system_message = "당신은 주어진 데이터를 분석하는 데이터 분석가입니다.\n"
    system_message += f"주어진 DataFrame에서 데이터를 출력하여 주어진 질문에 답할 수 있는 파이썬 코드를 작성하세요. "
    system_message += f"{df_name} DataFrame에 액세스할 수 있습니다.\n"
    system_message += f"`{df_name}` DataFrame에는 다음과 같은 열이 있습니다: {df_columns}\n"
    system_message += "데이터는 이미 로드되어 있으므로 데이터 로드 코드를 생략해야 합니다."

    message_with_data_info = [
        ("system", system_message),
        ("human", "{question}"),
    ]

    prompt_with_data_info = ChatPromptTemplate.from_messages(message_with_data_info)

    # 체인을 구성합니다.
    code_generate_chain = (
        {"question": RunnablePassthrough()}
        | prompt_with_data_info
        | llm 
        | StrOutputParser()
        | python_code_parser
    )
    code = code_generate_chain.invoke(question)
    answer = run_code(code)
    
    return {'code': code, 'data': answer}

## Node 생성
# Node는 그래프에서 실행될 수 있는 작업을 정의합니다.
# Node는 함수로 정의되며, StateGraph를 정의할 때 사용한 State type을 입력으로 받습니다.
# Node는 state를 업데이트하거나, 새로운 state를 반환할 수 있습니다.
def query(state: State):
    """
    데이터를 쿼리하는 코드를 생성하고, 실행하고, 그 결과를 포함한 State를 반환합니다.
    위 과정은 앞서 정의한 `find_data` 함수를 활용합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): 쿼리한 데이터를 포함한 새로운 State
    """
    print("---데이터 쿼리---") # 현재 상태를 확인하기 위한 Print문
    question = state["question"]

    # Retrieval
    data = find_data(question)
    return {"question": question, "code": data['code'], "data": data['data'], "generation": data['code']}

def answer_with_data(state: State):
    """
    쿼리한 데이터를 바탕으로 답변을 생성합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """
    print("---데이터 기반 답변 생성---") # 현재 상태를 확인하기 위한 Print문
    question = state["question"]
    data = state["data"]

    # 데이터를 바탕으로 질문에 대답하는 코드를 생성합니다.
    reasoning_system_message = "당신은 데이터를 바탕으로 질문에 답하는 데이터 분석가입니다.\n"
    reasoning_system_message += f"사용자가 입력한 데이터를 바탕으로, 질문에 대답하세요."

    reasoning_user_message = "데이터: {data}\n{question}"

    reasoning_with_data = [
        ("system", reasoning_system_message),
        ("human", reasoning_user_message),
    ]
    reasoning_with_data_chain = ChatPromptTemplate.from_messages(reasoning_with_data) | llm | StrOutputParser()
    
    # 대답 생성
    generation = reasoning_with_data_chain.invoke({"data": data, "question": question})
    return {"question": question, "code": state['code'], "data": data, "generation": generation}

def init_answer(state: State) -> str:
    """
    코드 생성 프롬프트를 활용해서, 코드를 생성해야 할지, 그냥 답하면 될 지 결정하고 답변합니다.
    이 답변을 활용해서 다른 함수에서 데이터 쿼리를 진행할 지, 바로 답변 생성을 진행할 지 결정합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """
    print("---데이터 추출 필요성 확인---") # 현재 상태를 확인하기 위한 Print문
    question = state["question"]
    llm = ChatOpenAI(
        api_key="ollama",
        model="mistral:7b",
        base_url="http://localhost:11434/v1",
        streaming=True
    )
    decide_system_message = "당신은 주어진 데이터를 분석하는 데이터 분석가입니다.\n"
    decide_system_message += f"주어진 DataFrame에서 데이터를 출력하여 주어진 질문에 답할 수 있는 파이썬 코드를 작성하세요. "
    decide_system_message += f"{df_name} DataFrame에 액세스할 수 있습니다.\n"
    decide_system_message += f"`{df_name}` DataFrame에는 다음과 같은 열이 있습니다: {df_columns}\n"
    decide_system_message += "데이터는 이미 로드되어 있으므로 데이터 로드 코드를 생략해야 합니다."
    decide_system_message += "주어진 질문이 데이터와 무관하다면, 파이썬 코드를 생성하지 말고 주어진 질문에 답하세요."
    decide_user_message = "{question}"
    decide_prompt = ChatPromptTemplate.from_messages([
        ("system", decide_system_message),
        ("human", decide_user_message)
    ])
    
    decide_chain = {"question": RunnablePassthrough()} | decide_prompt | llm
    
    generation = decide_chain.invoke(question)  
    return {"question": question, "generation": generation.content}
    
def answer(state: State):
    """
    데이터를 쿼리하지 않고 답변을 바로 생성합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """
    print("---답변 생성---") # 현재 상태를 확인하기 위한 Print문
    question = state["question"]
    
    return {"question": question, "generation": llm.invoke(question).content}   

`decide_query` 함수를 통해 분기 로직을 구현하고, 그래프에 노드와 간선, 조건부 간선을 등록합니다.

In [ ]:
def decide_query(state: State) -> str:
    generation = state["generation"]
    
    print(f"Response: {generation}")
    if "```python" in generation.lower():
        return "query" # "query" 노드로 이동
    else:
        return "answer" # "answer" 노드로 이동
## 그래프 구성

# 앞서 정의한 Node를 모두 추가합니다.
workflow.add_node("init_answer", init_answer)
workflow.add_node("query", query)
workflow.add_node("answer", answer)
workflow.add_node("answer_with_data", answer_with_data)

# 시작지점을 정의합니다.
workflow.set_entry_point("init_answer")

# 간선을 정의합니다.
# END는 종결 지점을 의미합니다.
workflow.add_edge("answer", END) # workflow.set_finish_point("answer")와 동일합니다.
workflow.add_edge("answer_with_data", END)
workflow.add_edge("query", "answer_with_data")

# 조건부 간선을 정의합니다.
# init_answer 노드의 답변을 바탕으로 decide_query 함수에서 query 또는 answer로 분기합니다.
workflow.add_conditional_edges(
    "init_answer",
    decide_query,
    # 어떤 노드로 이동할지 mapping합니다. 없어도 무방하지만, Graph의 가독성을 높일 수 있습니다.
    {
        "query": "query",
        "answer": "answer"
    }
)

그래프를 컴파일합니다.

In [ ]:
graph = workflow.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

아래 질문을 입력해서 결과가 잘 나오는지 확인해봅시다. 
- 예시 질문 (데이터 활용): Velocity가 가장 큰 데이터를 알려줘
- 예시 질문 (데이터 무관): 오늘 점심으로 뭐 먹을까?

In [ ]:
while True:
    question = input("질문을 입력해주세요 (종료를 원하시면 '종료'를 입력해주세요.): ")
    if question == "종료":
        break
    else:
        # graph.invoke 함수를 사용하여 그래프를 실행하고, 최종 결과를 반환합니다.
        # 답변 생성에는 약 1분 정도 소요됩니다.
        print("Assistant: ", graph.invoke({"question": ("user", question)})['generation'])

## 2. 현재 챗봇의 한계점
- 단순 데이터 활용 챗봇이 대응하기 어려운 질문과 이를 해결할 수 있는 방법을 논의합니다.
  
저희가 구성한 챗봇에게 데이터의 분포나 특징에 대해 질문했을 때는, 
데이터를 추출하는 코드를 실행해서 정확하게 대답할 수 있었습니다.

더 나아가서, 현재 데이터를 바탕으로 미래의 데이터를 예측해봅시다.

In [ ]:
prompt = "현재 데이터를 바탕으로 미래의 데이터를 예측해줘."
for event in graph.stream({"question": ("user", prompt)}):
    for value in event.values():
        print(value)
        print("Assistant:", value["generation"])

위 코드를 실행하면, 대부분 아래 결과 중 하나가 나타납니다.
1. 별도로 코드를 생성하지 않고, 머신러닝 활용 권장
2. sklearn (scikit-learn)이 포함된 머신러닝 코드를 생성하고 실행했으나, 학습 로그만 보여주고 미래의 데이터 예측값을 보여주지 못함.

앞서 이론 강의에서 살펴본 것 같이 전혀 근거없는 답변 (할루시네이션)을 하지 않는 이유는,<br>
시스템 프롬프트에 '데이터 분석가' 역할을 명시했고, 질문에 답할 수 있는 코드를 실행한 후 답변하라고 명시했기 떄문입니다.

그렇지만, 두 경우 모두 저희가 원하는 답변은 아닙니다. 

1번 유형의 답변에서 유추할 수 있듯이, 머신러닝 기법을 활용해서 주어진 데이터를 회귀분석 하면 미래의 데이터를 예측할 수 있을 것입니다.

정말 머신러닝을 활용해서 이 질문에 대해 답할 수 있는지 확인해봅시다.

## 3. 주어진 질문을 위한 머신러닝을 활용한 데이터 분석
- 머신러닝을 통해 질문에 답할 수 있는 분석 결과를 반환하는 모델을 학습할 수 있음을 이해합니다.

머신러닝 기반 회귀 분석을 통해 현재 잉크젯 데이터의 향후 추세를 예측할 수 있습니다.

### 데이터 분할
현재 데이터를 8:2로 나눠서 80%는 모델을 학습 시키는데 사용하고, 20%는 모델의 예측 정확도를 평가하는데 사용합니다.

In [ ]:
X = df_inkjet.drop(['PatternSize'], axis=1)
y = df_inkjet['PatternSize']

In [ ]:
X_train, X_test, y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state = 1)
X.shape
print('학습 데이터 :' ,X_train.shape)
print('테스트 데이터 : ', X_test.shape)

### LangChain 활용 코드 생성 Chain 구성

- 머신러닝 모델 학습 코드를 생성하는 체인을 구성합니다.

In [ ]:
def ml_code_gen(question: str) -> str:
    """입력한 질문을 위한 데이터 쿼리 코드를 생성하여 실행한 후 결과를 반환하는 Tool."""
        # 프롬프트를 정의합니다.
    system_message = "당신은 주어진 데이터를 분석하는 머신러닝 분석가입니다.\n"

    user_message = f"질문에 답하기 위해 scikit-learn을 통해서 회귀모델을 정의하고, 학습하고, 예측 결과를 출력하는 코드를 작성하세요. "
    user_message += f"학습 데이터는 X_train과 y_train을 사용하고, 테스트 데이터는 X_test와 y_test를 사용합니다."

    message_with_ml = [
        ("system", system_message),
        ("human", "{question}" + user_message),
    ]

    prompt_with_ml = ChatPromptTemplate.from_messages(message_with_ml)

    # 체인을 구성합니다.
    code_generate_chain = (
        {"question": RunnablePassthrough()}
        | prompt_with_ml
        | llm 
        | StrOutputParser()
        | python_code_parser
    )
    code = code_generate_chain.invoke(question)
    return code

In [ ]:
# MAPE: Mean Absolute Percetnage Error의 약자로, 실제 값과 예측 값의 차이를 백분율로 나타낸 지표입니다.
code = ml_code_gen("현재 데이터를 바탕으로 미래의 데이터를 예측하고, 그 정확도를 MAPE Metric으로 출력해줘.")
print(code)

In [ ]:
# 챗봇이 생성한 코드를 직접 복사하여 실행해주세요.
# X_train, X_test, y_train, y_test 변수에 새로운 값을 할당하는 코드는 지운 후 실행해주세요.



대부분의 경우, `LinearRegression()` 생성자를 사용해서 선형 회귀 모델을 만들고 사용하는 코드를 생성할 것입니다.

그러나, 이 모델을 실행했을 때 MAPE Metric은 2 (200%) 정도로 매우 크게 나타날 것입니다. 그렇다면, 이 방법을 사용할 수 없는걸까요?

이어지는 7\~8 챕터에서 머신러닝을 학습하고, 9\~10 챕터에서 이 방법을 적용하기 위해 다양한 고도화를 진행해 봅시다.